# EyeAI — Prepare Local Qwen Models on Kaggle

This notebook downloads the public chat and embedding models once, validates them, and saves a reusable Kaggle output.

Outputs:

```text
/kaggle/working/eyeai_language_models/
├── qwen3_4b_instruct_2507/
├── qwen3_embedding_0_6b/
└── language_models_manifest.json
```

No EyeAI model training occurs here.

## 1. Paths and model identifiers

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

OUTPUT_ROOT = Path('/kaggle/working/eyeai_language_models')
CHAT_MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
EMBEDDING_MODEL_ID = 'Qwen/Qwen3-Embedding-0.6B'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Output root:', OUTPUT_ROOT)

## 2. Install the downloader and configure optional Hugging Face authentication

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'huggingface-hub>=0.30,<1'],
    check=True,
)

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('Hugging Face authentication configured.')
else:
    print('HF_TOKEN is not required for these public model repositories.')

## 3. Download both model snapshots

In [ ]:
from huggingface_hub import snapshot_download

chat_dir = OUTPUT_ROOT / 'qwen3_4b_instruct_2507'
embedding_dir = OUTPUT_ROOT / 'qwen3_embedding_0_6b'

snapshot_download(
    repo_id=CHAT_MODEL_ID,
    local_dir=chat_dir,
    token=hf_token,
)
snapshot_download(
    repo_id=EMBEDDING_MODEL_ID,
    local_dir=embedding_dir,
    token=hf_token,
)

manifest = {
    'chat_model': CHAT_MODEL_ID,
    'chat_model_dir': str(chat_dir),
    'embedding_model': EMBEDDING_MODEL_ID,
    'embedding_model_dir': str(embedding_dir),
}
(OUTPUT_ROOT / 'language_models_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)
print(json.dumps(manifest, indent=2))

## 4. Validate the saved model files

In [ ]:
required_chat = ['config.json', 'tokenizer_config.json']
required_embedding = ['config.json', 'tokenizer_config.json']

for name in required_chat:
    assert (chat_dir / name).is_file(), chat_dir / name
for name in required_embedding:
    assert (embedding_dir / name).is_file(), embedding_dir / name

chat_weight_files = list(chat_dir.glob('*.safetensors'))
embedding_weight_files = list(embedding_dir.glob('*.safetensors'))
assert chat_weight_files, 'No chat-model safetensors were found.'
assert embedding_weight_files, 'No embedding-model safetensors were found.'

for label, directory in [('Chat model', chat_dir), ('Embedding model', embedding_dir)]:
    total_gib = sum(path.stat().st_size for path in directory.rglob('*') if path.is_file()) / (1024 ** 3)
    print(f'{label}: {directory}')
    print(f'  Size: {total_gib:.3f} GiB')

print()
print('Save this Notebook with output, then attach the output as a Kaggle Input.')